# ALS Collaborative Filtering - Model Training
### Based on: Padhy, N. et al. (2024) - A Recommendation System for E-Commerce Products Using Collaborative Filtering Approaches
### Journal: Engineering Proceedings (MDPI), Vol. 67, No. 1, Article 50
### DOI: https://doi.org/10.3390/engproc2024067050

In [0]:
from pyspark.sql.functions import col, count, lit, months_between, current_date
from pyspark.sql.functions import max as spark_max
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
import mlflow
import mlflow.spark

mlflow.autolog(disable=True)

print("imports done")

**Load tables**

In [0]:
orders_df  = spark.table("bharatmart.silver.slv_orders")
cart_df = spark.table("bharatmart.silver.slv_cart_events")
reviews_df = spark.table("bharatmart.silver.slv_reviews")

print(f"orders : {orders_df.count():,}")
print(f"cart : {cart_df.count():,}")
print(f"reviews : {reviews_df.count():,}")

**Clean Orders - Drop Nulls**
- Drop null customer_id and product_id

In [0]:
# EDA found 32,284 null customer_id + 55,000 null product_id
clean_orders = orders_df.filter(
    col("customer_id").isNotNull() & col("product_id").isNotNull()
)

print(f"before : {orders_df.count():,}")
print(f"after  : {clean_orders.count():,}")
print(f"dropped: {orders_df.count() - clean_orders.count():,}")

**Clean Reviews - Drop Ghost Orders**
- Drop ghost reviews, keep only high ratings
- ghost reviews have no real customer - flagged in silver layer
- ratings 1-2 are negative signals, 3 is borderline - both excluded

In [0]:
clean_reviews = reviews_df.filter(
    (col("_is_ghost_order") == False) & (col("rating") >= 4)
)

print(f"before : {reviews_df.count():,}")
print(f"after : {clean_reviews.count():,}")
print(f"dropped: {reviews_df.count() - clean_reviews.count():,}")

**Build Interaction Matrix**
- purchases at confidence 1.0 - money was spent, strongest implicit signal
- cart_add and wishlist_add at confidence 0.5 - intent but no purchase
- high reviews (4-5 star) at confidence 2.0 - bought and explicitly liked it
- take max confidence per user-item pair - same pair from multiple sources gets best signal

**Purchase signal**

In [0]:
purchase_df = clean_orders.select(
    col("customer_id"),
    col("product_id"),
    lit(1.0).alias("confidence")
)

print(f"purchase pairs : {purchase_df.count():,}")

**Cart signal**

In [0]:
cart_signal_df = cart_df.filter(
    col("action").isin("cart_add", "wishlist_add")
).select(
    col("customer_id"),
    col("product_id"),
    lit(0.5).alias("confidence")
)

print(f"cart signal pairs : {cart_signal_df.count():,}")

**Review signal**

In [0]:
review_signal_df = clean_reviews.select(
    col("customer_id"),
    col("product_id"),
    lit(2.0).alias("confidence")
)

print(f"review signal pairs : {review_signal_df.count():,}")

### **Merge All Signals**

**Union and take max confidence per pair**

In [0]:
all_signals = purchase_df.union(cart_signal_df).union(review_signal_df)

interaction_df = all_signals.groupBy("customer_id", "product_id") \
    .agg(spark_max("confidence").alias("confidence"))

print(f"total unique user-item pairs : {interaction_df.count():,}")

**Verify confidence distribution**

In [0]:
interaction_df.groupBy("confidence") \
    .count() \
    .orderBy("confidence") \
    .show()

**StringIndexer - Map String IDs to Integers**
- ALS requires integer user and item IDs
- customer_id and product_id are string UUIDs in BharatMart
- StringIndexer maps them to sequential integers 0, 1, 2...
- indexer_model saved to MLflow so scoring notebook can reverse the mapping

**Fit StringIndexer**

In [0]:
user_indexer = StringIndexer(inputCol="customer_id", outputCol="user_idx")
item_indexer = StringIndexer(inputCol="product_id",  outputCol="item_idx")

indexer_pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexer_model = indexer_pipeline.fit(interaction_df)
indexed_df = indexer_model.transform(interaction_df)

print(f"indexed rows : {indexed_df.count():,}")

**Verify integer IDs look correct**

In [0]:
display(indexed_df.select(
    "customer_id", "user_idx",
    "product_id", "item_idx",
    "confidence"
).limit(5))

**Train/Test Split - Temporal 70/30**
- train on first 70% of interactions by date, test on last 30%
- temporal split prevents data leakage - model cannot see future purchases during training
- Padhy et al. (2024) used random split but temporal is more realistic for production

**Add order date to indexed_df for temporal split**

In [0]:
order_dates = clean_orders.select("customer_id", "product_id", "order_date")

indexed_with_date = indexed_df.join(
    order_dates, ["customer_id", "product_id"], "left"
)

print(f"rows after date join : {indexed_with_date.count():,}")

**Find the 70th percentile date cutoff**

In [0]:
from pyspark.sql.functions import percentile_approx

cutoff = indexed_with_date.select(
    percentile_approx("order_date", 0.7).alias("cutoff_date")
).toPandas()

cutoff_date = cutoff["cutoff_date"][0]
print(f"train/test cutoff date : {cutoff_date}")

**Fix Duplicate Rows from Date Join**
- 428 user-item pairs had repeat purchases - each with a different order date
- the join created 428 duplicate rows
- keep most recent order date per pair - most recent purchase is most relevant signal

**Deduplicate by keeping latest order date per pair**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("customer_id", "product_id").orderBy(col("order_date").desc())

indexed_deduped = indexed_with_date.withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"before dedup : {indexed_with_date.count():,}")
print(f"after dedup : {indexed_deduped.count():,}")

**Split into train and test**

In [0]:
train_df = indexed_deduped.filter(col("order_date") <= cutoff_date)
test_df  = indexed_deduped.filter(col("order_date") >  cutoff_date)

print(f"train rows : {train_df.count():,}")
print(f"test rows : {test_df.count():,}")
print(f"train % : {train_df.count()/indexed_deduped.count()*100:.1f}%")

**ALS Model Training with CrossValidator**
- rank: number of latent factors each user and item gets
- regParam: regularisation strength - Padhy et al. (2024) explicitly warn ALS overfits without this
- maxIter: how many alternating solve iterations to run
- implicitPrefs=True: treats confidence as weight not rating - correct for purchase data
- coldStartStrategy=drop: avoids NaN predictions for unseen users/items in test set

**Define ALS and parameter grid**

In [0]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

als = ALS(
    userCol="user_idx",
    itemCol="item_idx",
    ratingCol="confidence",
    implicitPrefs=True,
    coldStartStrategy="drop",
    alpha=40.0,
    seed=42
)

param_grid = ParamGridBuilder() \
    .addGrid(als.rank,     [10, 20, 50]) \
    .addGrid(als.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(als.maxIter,  [10, 20]) \
    .build()

print(f"total parameter combinations : {len(param_grid)}")

**Run CrossValidator**

In [0]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="confidence",
    predictionCol="prediction"
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print("starting cross validation - this will take a few minutes...")
cv_model = cv.fit(train_df)
print("cross validation done")

**Best model params**

In [0]:
best_model = cv_model.bestModel

print(f"best rank : {best_model.rank}")
print(f"best regParam : {best_model._java_obj.parent().getRegParam()}")
print(f"best maxIter : {best_model._java_obj.parent().getMaxIter()}")

**RMSE on test set**

In [0]:
predictions = best_model.transform(test_df)

rmse = evaluator.evaluate(predictions)
print(f"test RMSE : {rmse:.4f}")

**Generate Top-10 recommendations for all users**
- recommendForAllUsers returns top N product recommendations per user

In [0]:
recs_df = best_model.recommendForAllUsers(10)
print(f"users with recommendations : {recs_df.count():,}")

**Compute Precision@10**

In [0]:
from pyspark.sql.functions import explode, col

# explode recommendations into one row per user-item pair
recs_exploded = recs_df.select(
    col("user_idx"),
    explode(col("recommendations")).alias("rec")
).select(
    col("user_idx"),
    col("rec.item_idx").alias("item_idx"),
    col("rec.rating").alias("als_score")
)

# actual purchases in test set
actual = test_df.select("user_idx", "item_idx").distinct()

# how many recommended items were actually purchased
hits = recs_exploded.join(actual, ["user_idx", "item_idx"], "inner").count()
total_recs = recs_exploded.count()

precision_at_10 = hits / recs_df.count()
print(f"hits : {hits:,}")
print(f"total recs : {total_recs:,}")
print(f"precision@10  : {precision_at_10:.4f}")

**Log to MLflow**

In [0]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_schema = Schema([
    ColSpec("integer", "user_idx"),
    ColSpec("integer", "item_idx"),
    ColSpec("double",  "confidence")
])

output_schema = Schema([
    ColSpec("double", "prediction")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="als_implicit_cv") as run:
    mlflow.log_param("rank",          best_model.rank)
    mlflow.log_param("regParam",      best_model._java_obj.parent().getRegParam())
    mlflow.log_param("maxIter",       best_model._java_obj.parent().getMaxIter())
    mlflow.log_param("implicitPrefs", True)
    mlflow.log_param("alpha",         40.0)
    mlflow.log_param("train_size",    train_df.count())
    mlflow.log_param("test_size",     test_df.count())
    mlflow.log_param("paper_warning", "ALS overfits without regParam tuning - Padhy et al. 2024")

    mlflow.log_metric("rmse",                rmse)
    mlflow.log_metric("precision_at_10",     precision_at_10)
    mlflow.log_metric("total_interactions",  1600485)
    mlflow.log_metric("cold_start_customers", 2692)

    mlflow.spark.log_model(
        best_model,
        artifact_path="als_model",
        registered_model_name="bharatmart.ml.bharatmart_als_model",
        signature=signature
    )

    run_id = run.info.run_id
    print(f"run_id : {run_id}")
    print("mlflow logging done")

**Register model alias as Production**

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Unity Catalog uses search_model_versions instead of latest_versions
versions = client.search_model_versions("name='bharatmart.ml.bharatmart_als_model'")
latest_version = versions[0].version

client.set_registered_model_alias(
    name="bharatmart.ml.bharatmart_als_model",
    alias="Production",
    version=latest_version
)

print(f"model version : {latest_version}")
print(f"alias : Production")
print("registration done")

**Final summary print**

In [0]:
print("=== ALS Model Training Complete ===")
print(f"best rank : {best_model.rank}")
print(f"best regParam : {best_model._java_obj.parent().getRegParam()}")
print(f"test RMSE : {rmse:.4f}")
print(f"precision@10 : {precision_at_10:.4f}")
print(f"run_id : {run_id}")
print(f"model : bharatmart.ml.bharatmart_als_model@Production")
print("===================================")

ALS CrossValidator tested 18 parameter combinations across 3 folds.
best params: rank=10, regParam=0.01, maxIter=10.

all three are the minimum values tested. on a 99.96% sparse binary 
matrix with 47,030 products, simpler models generalise better. higher 
rank means more latent dimensions per user and item with only 17 
interactions per customer on average there is not enough signal to 
fill those dimensions meaningfully.

regParam=0.01 is the lowest regularisation tested. Padhy et al. (2024) 
warned that ALS overfits without proper hyperparameter handling that 
warning applied to their explicit rating setup with higher confidence 
variance. on a near-binary confidence matrix the overfitting risk is 
lower and the model confirms this by preferring minimal regularisation.

test RMSE: 1.2003. lower than the paper's ALS RMSE of 1.4485 on 
explicit ratings, but the two are not directly comparable, we are in 
implicit mode predicting confidence weights, not star ratings.

precision@10: 0.0019. of the top 10 recommendations per customer, 
0.19% were actually purchased in the holdout period. random chance on 
a 47,030 product catalog is 0.02% - the model is 9.5x better than 
random. precision@10 is intentionally conservative here because the 
test set captures only 30% of purchase history. a customer who buys 
running shoes in the test period may have already bought protein powder 
and yoga mats in training ALS correctly recommends those but they 
don't appear as hits in the test set because they were already purchased.

model registered: bharatmart.ml.bharatmart_als_model@Production
run_id: 16545870b43d471b92e3a292a67e5096